# 03 · Free open judge + grader (calibration and grade impact)
Team Dropouts · Correction Inception

## Free open judge + grader (no paid API)
Settings: **Accelerator = GPU T4 x2**, **Internet = On**. Add Inputs (private Kaggle Datasets): `lma_src.zip` and `edu_text.zip` (from `D:/lma/kaggle_upload`). No HF token needed.

Judge/grader = **Qwen2.5-7B-Instruct (4-bit)**, batched on both T4s; our scripts call it through the hf-text backend. Because the paper's Gemini judge verdicts ship with the data, we **calibrate** this open judge against them (agreement / Cohen's kappa).

In [ ]:
import os, glob, shutil, zipfile, sys
IN = os.environ.get('KAGGLE_INPUT', '/kaggle/input')
WORK = os.environ.get('KAGGLE_WORK', '/kaggle/working')
def find(pattern):
    hits = sorted(glob.glob(f'{IN}/**/{pattern}', recursive=True) + glob.glob(f'{WORK}/_in/**/{pattern}', recursive=True))
    return hits[0] if hits else None
# Kaggle usually unzips uploads; if it kept a .zip, extract it ourselves
if not (find('src/eval/llm.py') and find('edu_data')):
    for z in glob.glob(f'{IN}/**/*.zip', recursive=True):
        zipfile.ZipFile(z).extractall(f'{WORK}/_in/' + os.path.basename(z)[:-4])
llm, EDU_DATA, EDU_REPO = find('src/eval/llm.py'), find('edu_data'), find('edu_repo')
if not (llm and EDU_DATA and EDU_REPO):
    for root, dirs, files in os.walk(IN):
        if root.count(os.sep) - IN.count(os.sep) < 4: print(root, dirs[:5], files[:5])
    raise SystemExit('Missing inputs: attach BOTH datasets (lma-src and edu-text) via Add Input. Found: '
                     + str(dict(src=llm, edu_data=EDU_DATA, edu_repo=EDU_REPO)))
shutil.copytree(os.path.dirname(os.path.dirname(llm)), f'{WORK}/src', dirs_exist_ok=True)
os.environ.update(EDU_DATA=EDU_DATA, EDU_REPO=EDU_REPO)
os.chdir(WORK)
# Re-runs: attach the previous run's output as an input; its answer caches are reused (failed calls aren't cached)
for rel in ['results/llm_cache.sqlite', 'results/judge_calibration/llm_cache.sqlite']:
    old = find(rel)
    if old and not os.path.exists(f'{WORK}/{rel}'):
        os.makedirs(os.path.dirname(f'{WORK}/{rel}'), exist_ok=True); shutil.copy(old, f'{WORK}/{rel}'); print('reusing cache', old)
print('code:', llm, '\ndata:', EDU_DATA, '\nrepo:', EDU_REPO)
if shutil.which('nvidia-smi'): os.system('nvidia-smi --query-gpu=name,memory.total --format=csv')

### Judge = Qwen2.5-7B-Instruct, 4-bit, batched transformers (one copy per T4; use **GPU T4 x2**).
No vLLM: its current wheels don't install cleanly on Kaggle's T4 image.

In [ ]:
if not os.environ.get('DRY_JUDGE'):
    os.system(f'{sys.executable} -m pip install -q -U accelerate bitsandbytes python-dotenv')
os.environ.update(HF_BATCH='8', HF_TOKEN_BUDGET='16000')  # lower the budget if you see 'CUDA out of memory'
JUDGE = os.environ.get('DRY_JUDGE') or 'hf-text:Qwen/Qwen2.5-7B-Instruct'
WORKERS = 16
LIMIT = os.environ.get('DRY_LIMIT', '')  # dry runs only: '--limit 5'
print('judge =', JUDGE, '| workers =', WORKERS)

## 1. Judge calibration: open judge vs the paper's Gemini judge on the SAME Gemini-2.5-Pro transcripts (513 samples)

In [ ]:
!python src/eval/evaluate_recognition.py --transcripts-root {EDU_DATA}/Observationset_Final --task v6_Gemini_2p5 --model-dir gemini-2.5-pro --judge {JUDGE} --workers {WORKERS} {LIMIT} --out {WORK}/results/judge_calibration
!python src/eval/calibrate_judge.py --ours {WORK}/results/judge_calibration/v6_Gemini_2p5/step10_rechecked.csv --shipped gemini-2.5-pro

## 2. Grade impact (B1): does transcription change the grade? Same open grader on expert vs model transcripts
Also reports grader validity vs TA labels and the grader noise floor (`--repeats 1`).

In [ ]:
for folder, mdir in [('v6_Gemini_2p5', 'gemini-2.5-pro'), ('v6_Gemini_3', 'gemini-3-pro-preview'), ('v6_gpt5p1', 'gpt-5.1')]:
    !python src/eval/grade_impact.py --target-root {EDU_DATA}/Observationset_Final/{folder} --target-model-dir {mdir} --grader {JUDGE} --name B1_{folder}_qwen7b --repeats 1 --workers {WORKERS} {LIMIT}

In [ ]:
shutil.make_archive(f'{WORK}/open_judge_results', 'zip', WORK, 'results')
print(os.path.getsize(f'{WORK}/open_judge_results.zip') // 1024, 'KB')